# Pixel Transformer tutorial v2: choosing a model route

## First, a few words on the goal of this tutorial

In the BDT notebook, we classified pixel clusters using engineered cluster quantities. Here we will classify the same objects using the raw pixel hits inside each cluster.

The goal is not to memorize a perfect Transformer recipe. The goal is to understand the moving pieces well enough to make one intentional design choice, test it, and compare it to a baseline.

There are two ways to use this notebook:

1. Pick one of the known routes. You fill in the relevant architecture lines, and the notebook loads matching pretrained weights.
2. Pick `custom`. You define your own model and train it from scratch.

Known routes never train in this notebook. If their pretrained weights are missing, the notebook raises an error rather than silently starting a long training job.


## Outline

1. Load the raw-hit pixel-cluster data.
2. Build a small Transformer from familiar PyTorch layers.
3. Choose a model route.
4. Load the pretrained route, or train only if you chose `custom`.
5. Compare against the BDT reference.


## Setup

Most of the plotting, benchmarking, and data-loading code lives in the parent `tutorial/` directory. This notebook adds only the route-specific model choices.


In [ ]:
import sys, os, json, math, time, random
from pathlib import Path

# Find the ML4FPS checkout from the notebook location.
PROJECT_ROOT = Path.cwd().resolve()
PRETRAINED_DIR = PROJECT_ROOT / 'simple_transformer_solution_outputs'
OUTPUT_DIR = PROJECT_ROOT / 'custom_model_outputs'

# Use the tutorial-local utilities and dataloader.
path_str = str(PROJECT_ROOT)
if path_str in sys.path:
    sys.path.remove(path_str)
sys.path.insert(0, path_str)

import numpy as np
import torch
import torch.nn as nn

from dataloader import load_pixel_data
from tutorial_utils import (
    plot_confusion_matrix, plot_roc_comparison,
    count_parameters, display_model_card,
    load_pretrained_model, make_context_loaders, route_uses_context,
    build_model_for_route, load_pretrained_route, train_custom_model,
    compare_bdt_and_route, resolve_route_config,
)


## Configuration

We keep the same split and hit cap used to produce the pretrained solution weights. Changing these values is a real experiment, but it means the saved weights are no longer a clean reference.


In [ ]:
SIGNAL_H5 = str('/global/cfs/cdirs/m5197/sferrar2/ML4FPS/Samples_v2/signal.h5')
BIB_H5    = str('/global/cfs/cdirs/m5197/sferrar2/ML4FPS/Samples_v2/BIB.h5')

MAX_RAW_HITS = 50
TEST_FRACTION = 0.2
VAL_FRACTION  = 0.1
RANDOM_SEED   = 42
BATCH_SIZE    = 256

BDT_MODEL_PATH = PROJECT_ROOT / 'bdt_baseline_outputs' / 'bdt_gradient_boosting.joblib'

for required_path in [SIGNAL_H5, BIB_H5]:
    if not Path(required_path).exists():
        raise FileNotFoundError(required_path)


In [ ]:
def get_device():
    device = 'cpu'
    if torch.cuda.is_available():
        try:
            torch.zeros(1).cuda()
            device = 'cuda'
        except RuntimeError:
            print('CUDA device found but not usable. Falling back to CPU.')
    return device


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(RANDOM_SEED)
device = get_device()
print('Using device:', device)


## Load Raw-Hit Data

Each cluster is represented as a padded sequence of raw pixel hits. A hit has four primitive features: energy, time, local x, and local y. Empty padded slots are all zeros.


In [ ]:
data = load_pixel_data(
    SIGNAL_H5, BIB_H5,
    max_hits=MAX_RAW_HITS,
    batch=BATCH_SIZE,
    test_frac=TEST_FRACTION,
    val_frac=VAL_FRACTION,
    seed=RANDOM_SEED,
)

train_loader = data['train']
val_loader   = data['val']
test_loader  = data['test']

idx_train = np.asarray(data['idx_train'])
idx_val   = np.asarray(data['idx_val'])
idx_test  = np.asarray(data['idx_test'])
labels    = np.asarray(data['labels']).astype(int)
CLUSTER_FEATURES = list(data['feature_keys'])

print(f"Total clusters: {len(labels):,}")
print(f"Signal: {labels.sum():,}, BIB: {(labels == 0).sum():,}")
print(f"Train: {len(idx_train):,}, Val: {len(idx_val):,}, Test: {len(idx_test):,}")

## The Base Transformer

The base model follows a simple pattern:

1. Project each hit into a hidden representation.
2. Let hits exchange information through self-attention.
3. Mask padded hit slots so they do not contribute.
4. Pool the learned hit representations into one cluster representation.
5. Classify the cluster as BIB or signal.

This is deliberately compact. The route exercises below change one design choice at a time.


In [ ]:
class Attention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.q = nn.Linear(dim, dim)
        self.k = nn.Linear(dim, dim)
        self.v = nn.Linear(dim, dim)
        self.scale = dim ** -0.5

    def forward(self, x, mask):
        q = self.q(x)
        k = self.k(x)
        v = self.v(x)

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        x = attn @ v

        return x * mask


class TransformerBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.att = Attention(dim)
        self.proj1 = nn.Linear(dim, dim)
        self.proj2 = nn.Linear(dim, dim)
        self.activation = nn.GELU()
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x, mask):
        x = x + self.att(self.norm1(x), mask)
        x = self.activation(self.proj1(x)) * mask
        x = x + self.proj2(self.norm2(x)) * mask
        return x


class SimplePixelTransformer(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=64, num_layers=3, num_classes=2):
        super().__init__()
        self.input_layer = nn.Linear(input_dim, hidden_dim)
        self.blocks = nn.ModuleList([TransformerBlock(hidden_dim) for _ in range(num_layers)])
        self.output_layer = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        # A hit is real if the energy feature is nonzero.
        mask = (x[:, :, 0] != 0).unsqueeze(-1).float()

        x = self.input_layer(x) * mask
        for block in self.blocks:
            x = block(x, mask)

        # Sum pooling keeps some occupancy information; the sqrt keeps the scale moderate.
        x = (x * mask).sum(dim=1) / (mask.shape[1] ** 0.5)
        return self.output_layer(x)


## The First Benchmark: Why Improve The Transformer?

Before choosing a route, let's look at the plain `SimplePixelTransformer`. It uses the architecture above and loads the pretrained reference weights from `simple_transformer_solution_outputs/simple_pixel_transformer_reference.pt`.

The point of this benchmark is motivational: the first Transformer is compact and direct, but it should lag behind the BDT reference. The route exercises below are ways to close that gap.


In [ ]:
default_config = {
    'input_dim': 4,
    'hidden_dim': 64,
    'num_layers': 3,
    'num_classes': 2,
}

DEFAULT_TRANSFORMER_PATH = PRETRAINED_DIR / 'simple_pixel_transformer_reference.pt'

default_transformer = SimplePixelTransformer(**default_config)
default_transformer = load_pretrained_model(
    default_transformer,
    DEFAULT_TRANSFORMER_PATH,
    device=device,
    name='SimplePixelTransformer reference',
)
display_model_card(default_transformer, default_config)

initial_comparison = compare_bdt_and_route(
    data=data,
    labels=labels,
    idx_test=idx_test,
    bdt_model_path=BDT_MODEL_PATH,
    model=default_transformer,
    route='default',
    model_name='SimplePixelTransformer reference',
    device=device,
    test_loader=test_loader,
)

plot_roc_comparison(initial_comparison['roc_payload'])
for name, (y, scores) in initial_comparison['score_payloads'].items():
    plot_confusion_matrix(y, scores, cut=0.5, title=f'{name} confusion matrix, cut = 0.5')


## Choose A Route

Set `MODEL_ROUTE` to one of the pretrained routes, or to `custom`.

The known routes are:

- `capacity`: widen the hidden representation.
- `depth`: use more Transformer blocks.
- `pooling`: combine mean and sum pooled hit features.
- `dropout`: add light dropout.
- `preprocess`: transform raw energy and time before the input layer.
- `context_head`: add detector context after hit pooling.
- `context_broadcast`: add detector context to every hit before attention.
- `custom`: train your own `MyPixelTransformer` from scratch.


In [ ]:
# Choose one route to try.
# Known routes load pretrained weights. The custom route trains your own model from scratch.
MODEL_ROUTE = 'context_broadcast'

KNOWN_PRETRAINED_ROUTES = [
    'capacity',
    'depth',
    'pooling',
    'dropout',
    'preprocess',
    'context_head',
    'context_broadcast',
]

if MODEL_ROUTE not in KNOWN_PRETRAINED_ROUTES + ['custom']:
    raise ValueError(f"Unknown MODEL_ROUTE: {MODEL_ROUTE}")

print('Selected route:', MODEL_ROUTE)


## Route 1: Capacity And Depth

Capacity asks: how much information can each hit representation carry?

Depth asks: how many rounds of hit-to-hit communication do we allow?

These two changes reuse the base class; the main exercise is to choose the config values that match the pretrained route you selected.


In [ ]:
class CapacityPixelTransformer(SimplePixelTransformer):
    pass


capacity_config = {
    'input_dim': 4,
    'hidden_dim': 128,
    'num_layers': 3,
    'num_classes': 2,
}


class DeeperPixelTransformer(SimplePixelTransformer):
    pass


depth_config = {
    'input_dim': 4,
    'hidden_dim': 64,
    'num_layers': 4,
    'num_classes': 2,
}


## Route 2: Pooling

Pooling is where a sequence of hits becomes one cluster-level vector. Sum pooling keeps occupancy information, while mean pooling asks what a typical hit looks like. This route gives the classifier both.


In [ ]:
class MeanSumPoolingPixelTransformer(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=64, num_layers=3, num_classes=2):
        super().__init__()
        self.input_layer = nn.Linear(input_dim, hidden_dim)
        self.blocks = nn.ModuleList([TransformerBlock(hidden_dim) for _ in range(num_layers)])
        self.output_layer = nn.Sequential(
            nn.LayerNorm(2 * hidden_dim),
            nn.Linear(2 * hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        mask = (x[:, :, 0] != 0).unsqueeze(-1).float()

        x = self.input_layer(x) * mask
        for block in self.blocks:
            x = block(x, mask)

        n_hits = mask.sum(dim=1).clamp(min=1)
        x_sum = (x * mask).sum(dim=1)
        x_mean = x_sum / n_hits
        x_sum_scaled = x_sum / (mask.shape[1] ** 0.5)
        x = torch.cat([x_mean, x_sum_scaled], dim=-1)

        return self.output_layer(x)


pooling_config = {
    'input_dim': 4,
    'hidden_dim': 64,
    'num_layers': 3,
    'num_classes': 2,
}


## Route 3: Dropout

Dropout randomly removes small pieces of information during training. At evaluation time it is disabled, but the saved weights were trained with that extra noise, which can make the model less brittle.


In [ ]:
class DropoutAttention(nn.Module):
    def __init__(self, dim, dropout=0.05):
        super().__init__()
        self.q = nn.Linear(dim, dim)
        self.k = nn.Linear(dim, dim)
        self.v = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout)
        self.scale = dim ** -0.5

    def forward(self, x, mask):
        q = self.q(x)
        k = self.k(x)
        v = self.v(x)

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = self.dropout(attn.softmax(dim=-1))
        x = attn @ v

        return x * mask


class DropoutTransformerBlock(nn.Module):
    def __init__(self, dim, dropout=0.05):
        super().__init__()
        self.att = DropoutAttention(dim, dropout=dropout)
        self.proj1 = nn.Linear(dim, dim)
        self.proj2 = nn.Linear(dim, dim)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x, mask):
        x = x + self.att(self.norm1(x), mask)
        x = self.activation(self.proj1(x)) * mask
        x = x + self.dropout(self.proj2(self.norm2(x))) * mask
        return x


class DropoutPixelTransformer(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=64, num_layers=3, num_classes=2, dropout=0.05):
        super().__init__()
        self.input_layer = nn.Linear(input_dim, hidden_dim)
        self.blocks = nn.ModuleList([
            DropoutTransformerBlock(hidden_dim, dropout=dropout) for _ in range(num_layers)
        ])
        self.output_layer = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        mask = (x[:, :, 0] != 0).unsqueeze(-1).float()

        x = self.input_layer(x) * mask
        for block in self.blocks:
            x = block(x, mask)

        x = (x * mask).sum(dim=1) / (mask.shape[1] ** 0.5)
        return self.output_layer(x)


dropout_config = {
    'input_dim': 4,
    'hidden_dim': 64,
    'num_layers': 3,
    'num_classes': 2,
    'dropout': 0.05,
}


## Route 4: Energy And Time Preprocessing

Raw energy often has a long positive tail. Time can also span a wide range. Simple transforms can make those features easier for a small network to use.

The padding mask still needs to be built from the raw, untransformed energy values.


In [ ]:
class PreprocessedPixelTransformer(SimplePixelTransformer):
    def preprocess_hits(self, x):
        x = x.clone()
        x[:, :, 0] = torch.log1p(torch.clamp(x[:, :, 0], min=0))
        x[:, :, 1] = torch.asinh(x[:, :, 1])
        return x

    def forward(self, x):
        # Build the mask before preprocessing, while padded hits are still exactly zero.
        mask = (x[:, :, 0] != 0).unsqueeze(-1).float()

        x = self.preprocess_hits(x)
        x = self.input_layer(x) * mask
        for block in self.blocks:
            x = block(x, mask)

        x = (x * mask).sum(dim=1) / (mask.shape[1] ** 0.5)
        return self.output_layer(x)


preprocess_config = {
    'input_dim': 4,
    'hidden_dim': 64,
    'num_layers': 3,
    'num_classes': 2,
}


## Route 5: Primitive Detector Context

The raw local pixel coordinates do not fully describe where the cluster lives in the detector. The context routes add a few primitive cluster-position quantities: `cluster_x`, `cluster_y`, `cluster_z`, and `cluster_r`.

The context arrays use the same train/validation/test indices as the raw-hit loaders, so the splits stay aligned.


In [ ]:
CONTEXT_KEYS = ['cluster_x', 'cluster_y', 'cluster_z', 'cluster_r','cluster_size_x']
_CONTEXT_LOADERS = None


def get_context_loaders():
    global _CONTEXT_LOADERS
    if _CONTEXT_LOADERS is None:
        _CONTEXT_LOADERS = make_context_loaders(
            SIGNAL_H5,
            BIB_H5,
            CONTEXT_KEYS,
            train_loader,
            val_loader,
            test_loader,
            idx_train,
            idx_val,
            idx_test,
            BATCH_SIZE,
        )
    return _CONTEXT_LOADERS


In [ ]:
class ContextHeadPixelTransformer(SimplePixelTransformer):
    def __init__(self, input_dim=4, context_dim=4, hidden_dim=64, num_layers=3, num_classes=2, context_hidden_dim=16):
        super().__init__(input_dim=input_dim, hidden_dim=hidden_dim, num_layers=num_layers, num_classes=num_classes)
        self.context_layer = nn.Sequential(
            nn.LayerNorm(context_dim),
            nn.Linear(context_dim, context_hidden_dim),
            nn.GELU(),
        )
        self.output_layer = nn.Sequential(
            nn.LayerNorm(hidden_dim + context_hidden_dim),
            nn.Linear(hidden_dim + context_hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x, context):
        mask = (x[:, :, 0] != 0).unsqueeze(-1).float()

        x = self.input_layer(x) * mask
        for block in self.blocks:
            x = block(x, mask)

        x = (x * mask).sum(dim=1) / (mask.shape[1] ** 0.5)
        context = self.context_layer(context)
        x = torch.cat([x, context], dim=-1)
        return self.output_layer(x)


class ContextBroadcastPixelTransformer(SimplePixelTransformer):
    def __init__(self, input_dim=4, context_dim=4, hidden_dim=64, num_layers=3, num_classes=2, context_hidden_dim=16):
        super().__init__(input_dim=input_dim, hidden_dim=hidden_dim, num_layers=num_layers, num_classes=num_classes)
        self.context_layer = nn.Sequential(
            nn.LayerNorm(context_dim),
            nn.Linear(context_dim, context_hidden_dim),
            nn.GELU(),
        )
        self.input_layer = nn.Linear(input_dim + context_hidden_dim, hidden_dim)

    def forward(self, x, context):
        mask = (x[:, :, 0] != 0).unsqueeze(-1).float()

        context = self.context_layer(context)
        context = context.unsqueeze(1).expand(-1, x.shape[1], -1)
        x = torch.cat([x, context], dim=-1)

        x = self.input_layer(x) * mask
        for block in self.blocks:
            x = block(x, mask)

        x = (x * mask).sum(dim=1) / (mask.shape[1] ** 0.5)
        return self.output_layer(x)


context_config = {
    'input_dim': 4,
    'context_dim': len(CONTEXT_KEYS),
    'hidden_dim': 64,
    'num_layers': 3,
    'num_classes': 2,
    'context_hidden_dim': 16,
}

broadcast_context_config = dict(context_config)


## Custom Route

If you want to do something that does not match a pretrained architecture, set `MODEL_ROUTE = 'custom'` above and edit `MyPixelTransformer` below.

That is the only path in this notebook that trains from scratch. It saves into this v2 directory and never overwrites the pretrained solution weights.


In [ ]:
class MyPixelTransformer(PreprocessedPixelTransformer):
    # One filled-in custom example: reuse the preprocessing idea, but make the hidden representation wider.
    pass


my_config = {
    'input_dim': 4,
    'hidden_dim': 96,
    'num_layers': 3,
    'num_classes': 2,
}

CUSTOM_MODEL_NAME = 'my_preprocessed_wide_transformer'
CUSTOM_NUM_EPOCHS = 60
CUSTOM_PATIENCE = 8
CUSTOM_LR = 3e-4


## Route Helpers

These helpers make the route decision explicit:

- `build_model_for_route(route)` creates the right class.
- `load_pretrained_route(route, device)` loads saved weights for known routes.
- `train_custom_model(model, train_loader, val_loader, device)` trains only the custom route.
- `predict_route_model(model, route, device)` knows whether the route needs context.


In [ ]:
# The lambdas keep this registry lazy: only the selected route is instantiated.
MODEL_ROUTES = {
    'capacity': {
        'model_factory': lambda: CapacityPixelTransformer(**capacity_config),
        'config': lambda: capacity_config,
        'weights': 'capacity_transformer_reference.pt',
        'display_name': 'Capacity Transformer',
    },
    'depth': {
        'model_factory': lambda: DeeperPixelTransformer(**depth_config),
        'config': lambda: depth_config,
        'weights': 'depth_transformer_reference.pt',
        'display_name': 'Deeper Transformer',
    },
    'pooling': {
        'model_factory': lambda: MeanSumPoolingPixelTransformer(**pooling_config),
        'config': lambda: pooling_config,
        'weights': 'pooling_transformer_reference.pt',
        'display_name': 'Mean+Sum Pooling Transformer',
    },
    'dropout': {
        'model_factory': lambda: DropoutPixelTransformer(**dropout_config),
        'config': lambda: dropout_config,
        'weights': 'dropout_transformer_reference.pt',
        'display_name': 'Dropout Transformer',
    },
    'preprocess': {
        'model_factory': lambda: PreprocessedPixelTransformer(**preprocess_config),
        'config': lambda: preprocess_config,
        'weights': 'preprocess_transformer_reference.pt',
        'display_name': 'Preprocessed Transformer',
    },
    'context_head': {
        'model_factory': lambda: ContextHeadPixelTransformer(**context_config),
        'config': lambda: context_config,
        'weights': 'context_head_transformer_reference.pt',
        'display_name': 'Context Head Transformer',
    },
    'context_broadcast': {
        'model_factory': lambda: ContextBroadcastPixelTransformer(**broadcast_context_config),
        'config': lambda: broadcast_context_config,
        'weights': 'context_broadcast_transformer_reference.pt',
        'display_name': 'Context Broadcast Transformer',
    },
}


## Build The Selected Model

Known routes load from `simple_transformer_solution_outputs`. The custom route trains from scratch.


In [ ]:
context_loaders = get_context_loaders() if route_uses_context(MODEL_ROUTE) else None

if MODEL_ROUTE == 'custom':
    selected_model = build_model_for_route(
        'custom',
        MODEL_ROUTES,
        custom_model_class=MyPixelTransformer,
        custom_config=my_config,
    )
    selected_model = train_custom_model(
        selected_model,
        my_config,
        train_loader,
        val_loader,
        device=device,
        output_dir=OUTPUT_DIR,
        model_name=CUSTOM_MODEL_NAME,
        num_epochs=CUSTOM_NUM_EPOCHS,
        patience=CUSTOM_PATIENCE,
        lr=CUSTOM_LR,
    )
    selected_model_name = 'Custom MyPixelTransformer'
else:
    selected_model = load_pretrained_route(MODEL_ROUTE, MODEL_ROUTES, PRETRAINED_DIR, device=device)
    selected_model_name = MODEL_ROUTES[MODEL_ROUTE]['display_name']
    display_model_card(selected_model, resolve_route_config(MODEL_ROUTES[MODEL_ROUTE]))

print('Ready to evaluate:', selected_model_name)


## Compare Against The BDT Reference

The clearest comparison is visual: the ROC curve shows separation over all thresholds, and the confusion matrices show what happens at a fixed score cut.


In [ ]:
selected_comparison = compare_bdt_and_route(
    data=data,
    labels=labels,
    idx_test=idx_test,
    bdt_model_path=BDT_MODEL_PATH,
    model=selected_model,
    route=MODEL_ROUTE,
    model_name=selected_model_name,
    device=device,
    test_loader=test_loader,
    context_loaders=context_loaders,
)

plot_roc_comparison(selected_comparison['roc_payload'])
for name, (y, scores) in selected_comparison['score_payloads'].items():
    plot_confusion_matrix(y, scores, cut=0.5, title=f'{name} confusion matrix, cut = 0.5')


## Takeaway

A Transformer is not one single architecture. It is a set of choices about representation size, communication depth, pooling, regularization, preprocessing, and context. A lot of work has to go in to optimizing your transformer
